1. Класс «Товар» содержит следующие закрытые поля: \
● название товара \
● название магазина, в котором подаётся товар \
● стоимость товара в рублях \
Класс «Склад» содержит закрытый массив товаров. \
Обеспечить следующие возможности: \
● вывод информации о товаре со склада по индексу \
● вывод информации о товаре со склада по имени товара \
● сортировка товаров по названию, по магазину и по цене \
● перегруженная операция сложения товаров по цене

In [69]:
from typing import Iterable, Self
from dataclasses import dataclass


@dataclass(frozen=True, order=True)
class Product:
    name: str
    store_name: str
    price_rubles: float

    def __add__(self, other: Self) -> Self:
        return Product(f"{self.name}+{other.name}", f"{self.store_name}+{other.store_name}", self.price_rubles + other.price_rubles)

class Storage:
    def __init__(self, products: Iterable[Product] = ()):
        self._products: dict[str, Product] = {product.name: product for product in products}

    def __iter__(self) -> Iterable[Product]:
        return iter(self._products.values())

    def __getitem__(self, key: str | int) -> Product:
        if isinstance(key, int):
            return tuple(self._products.values())[key]

        return self._products[key]

    def __sort_by(self, by: str) -> tuple[Product]:
        list_products = list(self._products.values())
        list_products.sort(key=lambda product: product.__getattribute__(by))

        return tuple(list_products)

    def sort_by_product_name(self) -> tuple[Product]:
        return self.__sort_by("name")

    def sort_by_store_name(self) -> tuple[Product]:
        return self.__sort_by("store_name")

    def sort_by_price_rubles(self) -> tuple[Product]:
        return self.__sort_by("price_rubles")

products = [Product("hello", "hello_shop2", 1234), Product("hello2", "hello_shop3", 123), Product("hello3", "hello_shop1", 1233)]
storage = Storage(products)

print(storage["hello"])
print(storage[1])

print(products[0] + products[2])
print()
print(f"{storage.sort_by_product_name() = }\n------")
print(f"{storage.sort_by_store_name() = }\n------")
print(f"{storage.sort_by_price_rubles() = }\n------")


Product(name='hello', store_name='hello_shop2', price_rubles=1234)
Product(name='hello2', store_name='hello_shop3', price_rubles=123)
Product(name='hello+hello3', store_name='hello_shop2+hello_shop1', price_rubles=2467)

storage.sort_by_product_name() = (Product(name='hello', store_name='hello_shop2', price_rubles=1234), Product(name='hello2', store_name='hello_shop3', price_rubles=123), Product(name='hello3', store_name='hello_shop1', price_rubles=1233))
------
storage.sort_by_store_name() = (Product(name='hello3', store_name='hello_shop1', price_rubles=1233), Product(name='hello', store_name='hello_shop2', price_rubles=1234), Product(name='hello2', store_name='hello_shop3', price_rubles=123))
------
storage.sort_by_price_rubles() = (Product(name='hello2', store_name='hello_shop3', price_rubles=123), Product(name='hello3', store_name='hello_shop1', price_rubles=1233), Product(name='hello', store_name='hello_shop2', price_rubles=1234))
------


2. ПчёлоСлон \
Экземпляр класса инициализируется двумя целыми числами,
первое относится к пчеле, второе – к слону. Класс реализует
следующие методы: \
● fly() – возвращает True, если часть пчелы не меньше части
слона, иначе – False \
● trumpet() – если часть слона не меньше части пчелы,
возвращает строку “tu-tu-doo-doo”, иначе – “wzzzz” \
● eat(meal, value) – может принимать в meal только ”nectar”
или “grass”. Если съедает нектар, то value вычитается из
части слона, пчеле добавляется. Иначе – наоборот. Не
может увеличиваться больше 100 и уменьшаться меньше 0.

In [50]:
from warnings import warn
from typing import Callable, Any

class Beephant:
    def __init__(self, bee_part_percent: int = 50, elephant_part_percent: int = 50, error_function: Callable[[str], Any] = warn):
        if not isinstance(bee_part_percent, int) or not isinstance(elephant_part_percent, int):
            warn("bee_part_percent and elephant_part_percent must be integers")

        self.error_function: Callable[[str], Any] = error_function
        self.bee_part_percent: int = bee_part_percent
        self.elephant_part_percent: int = elephant_part_percent

        self._normalize_values()

    def _normalize_values(self):
        if self.elephant_part_percent < 0:
            self.elephant_part_percent = 0
            self.error_function("elephant_part_percent must be greater than 0. Elephant Part Percent is set to 0.")
        elif self.elephant_part_percent > 100:
            self.elephant_part_percent = 100
            self.error_function("elephant_part_percent must be less than 100. Elephant Part Percent is set to 100.")

        if self.bee_part_percent < 0:
            self.bee_part_percent = 0
            self.error_function("bee_part_percent must be greater than 0. Bee Part Percent is set to 0.")
        elif self.bee_part_percent > 100:
            self.bee_part_percent = 100
            self.error_function("bee_part_percent must be less than 100. Bee Part Percent is set to 100.")

        if self.bee_part_percent + self.elephant_part_percent > 100:
            if self.bee_part_percent > self.elephant_part_percent:
                self.elephant_part_percent = 100 - self.bee_part_percent
            else:
                self.bee_part_percent = 100 - self.elephant_part_percent

            self.error_function(f"bee_part_percent + elephant_part_percent must be less than 100. Now values are {self.elephant_part_percent} for elephant and {self.bee_part_percent} for bee.")

    def fly(self) -> bool:
        return self.bee_part_percent >= self.elephant_part_percent

    def trumpet(self) -> str:
        if self.elephant_part_percent >= self.bee_part_percent:
            return "tu-tu-doo-doo"

        return "wzzzz"

    def eat(self, meal: str, value: int) -> None:
        if meal == "nectar":
            self.bee_part_percent += value
            self.elephant_part_percent -= value
        elif meal == "grass":
            self.elephant_part_percent += value
            self.bee_part_percent -= value
        else:
            self.error_function("Meal must be 'nectar', 'grass'.")

        self._normalize_values()


In [51]:
"""TESTS FOR BEEPHANT (ChatGPT thanks)"""
from warnings import warn

def test_initialization():
    print("\n=== Тест инициализации ===")
    beephant = Beephant(40, 60)
    print(f"Bee Part: {beephant.bee_part_percent} (ожидалось 40)")
    print(f"Elephant Part: {beephant.elephant_part_percent} (ожидалось 60)")
    print("Результат:", beephant.bee_part_percent == 40 and beephant.elephant_part_percent == 60)

def test_initialization_over_100():
    print("\n=== Тест нормализации (сумма > 100) ===")
    beephant = Beephant(70, 50)
    total = beephant.bee_part_percent + beephant.elephant_part_percent
    print(f"Сумма процентов: {total} (ожидалось 100)")
    print("Результат:", total == 100)

def test_fly():
    print("\n=== Тест полета ===")
    beephant1 = Beephant(60, 40)
    print(f"Bee Part: {beephant1.bee_part_percent}, Elephant Part: {beephant1.elephant_part_percent}")
    print(f"Fly (ожидалось True): {beephant1.fly()}")

    beephant2 = Beephant(30, 70)
    print(f"Bee Part: {beephant2.bee_part_percent}, Elephant Part: {beephant2.elephant_part_percent}")
    print(f"Fly (ожидалось False): {beephant2.fly()}")

def test_trumpet():
    print("\n=== Тест звука ===")
    beephant1 = Beephant(30, 70)
    sound1 = beephant1.trumpet()
    print(f"Trumpet звук: {sound1} (ожидалось 'tu-tu-doo-doo')")

    beephant2 = Beephant(70, 30)
    sound2 = beephant2.trumpet()
    print(f"Trumpet звук: {sound2} (ожидалось 'wzzzz')")

def test_eat():
    print("\n=== Тест еды ===")
    beephant = Beephant(50, 50)

    print("\n--- Поедание нектара (добавляет пчелиную часть) ---")
    beephant.eat("nectar", 10)
    print(f"Bee Part: {beephant.bee_part_percent} (ожидалось 60), Elephant Part: {beephant.elephant_part_percent} (ожидалось 40)")

    print("\n--- Поедание травы (добавляет слоновую часть) ---")
    beephant.eat("grass", 20)
    print(f"Bee Part: {beephant.bee_part_percent} (ожидалось 40), Elephant Part: {beephant.elephant_part_percent} (ожидалось 60)")

    print("\n--- Поедание травы 100 (проверка граничного условия) ---")
    beephant.eat("grass", 100)
    print(f"Bee Part: {beephant.bee_part_percent} (ожидалось 0), Elephant Part: {beephant.elephant_part_percent} (ожидалось 100)")

    print("\n--- Поедание нектара 100 (проверка граничного условия) ---")
    beephant.eat("nectar", 100)
    print(f"Bee Part: {beephant.bee_part_percent} (ожидалось 100), Elephant Part: {beephant.elephant_part_percent} (ожидалось 0)")

def run_tests():
    test_initialization()
    test_initialization_over_100()
    test_fly()
    test_trumpet()
    test_eat()

run_tests()




=== Тест инициализации ===
Bee Part: 40 (ожидалось 40)
Elephant Part: 60 (ожидалось 60)
Результат: True

=== Тест нормализации (сумма > 100) ===
Сумма процентов: 100 (ожидалось 100)
Результат: True

=== Тест полета ===
Bee Part: 60, Elephant Part: 40
Fly (ожидалось True): True
Bee Part: 30, Elephant Part: 70
Fly (ожидалось False): False

=== Тест звука ===
Trumpet звук: tu-tu-doo-doo (ожидалось 'tu-tu-doo-doo')
Trumpet звук: wzzzz (ожидалось 'wzzzz')

=== Тест еды ===

--- Поедание нектара (добавляет пчелиную часть) ---
Bee Part: 60 (ожидалось 60), Elephant Part: 40 (ожидалось 40)

--- Поедание травы (добавляет слоновую часть) ---
Bee Part: 40 (ожидалось 40), Elephant Part: 60 (ожидалось 60)

--- Поедание травы 100 (проверка граничного условия) ---
Bee Part: 0 (ожидалось 0), Elephant Part: 100 (ожидалось 100)

--- Поедание нектара 100 (проверка граничного условия) ---
Bee Part: 100 (ожидалось 100), Elephant Part: 0 (ожидалось 0)


C:\Users\Liteop\AppData\Local\Temp\ipykernel_54052\2509005208.py:36: UserWarning: bee_part_percent + elephant_part_percent must be less than 100. Now values are 30 for elephant and 70 for bee.
  self.error_function(f"bee_part_percent + elephant_part_percent must be less than 100. Now values are {self.elephant_part_percent} for elephant and {self.bee_part_percent} for bee.")
C:\Users\Liteop\AppData\Local\Temp\ipykernel_54052\2509005208.py:21: UserWarning: elephant_part_percent must be less than 100. Elephant Part Percent is set to 100.
  self.error_function("elephant_part_percent must be less than 100. Elephant Part Percent is set to 100.")
C:\Users\Liteop\AppData\Local\Temp\ipykernel_54052\2509005208.py:25: UserWarning: bee_part_percent must be greater than 0. Bee Part Percent is set to 0.
  self.error_function("bee_part_percent must be greater than 0. Bee Part Percent is set to 0.")


3. Класс «Автобус». Класс содержит свойства: \
● скорость \
● максимальное количество посадочных мест \
● максимальная скорость \
● список фамилий пассажиров \
● флаг наличия свободных мест \
● словарь мест в автобусе \
Методы: \
● посадка и высадка одного или нескольких пассажиров \
● увеличение и уменьшение скорости на заданное значение \
● операции in, += и -= (посадка и высадка пассажира по \
фамилии)

In [65]:
class Bus:
    def __init__(self, *,
                    speed: float = 50,
                    max_speed: float = 90,
                    max_capacity: int = 100,
                    passengers_last_names: Iterable[str],
                    error_function: Callable[[str], Any] = warn,
                 ):
        self._passengers_last_names_seats: dict[int, str | None] = {seat_index: None for seat_index in range(max_capacity)}

        for index, passengers_last_name in enumerate(passengers_last_names):
            if index > max_capacity:
                error_function(f"Number of passengers is bigger than capacity: {index}/{max_capacity}. Extra passengers are gone.")
                break

            self._passengers_last_names_seats[index] = passengers_last_name

        self._speed: float = speed
        self._max_speed: float = max_speed
        self._max_capacity: int = max_capacity
        self._has_free_seats: bool = (max_capacity - len(passengers_last_names)) > 0

        self.error_function: Callable[[str], Any] = error_function

        self._check_values()

    def _check_values(self) -> None:
        if self._speed > self._max_speed:
            self.error_function(f"Speed is too high {self._speed}/{self._max_speed}. Speed was set to {self._max_speed}")
            self._speed = self._max_speed
        elif self._speed < 0:
            self.error_function(f"{self._speed} is negative. Speed was set to 0.")
            self._speed = 0

        number_of_passengers = len(self._get_free_seats_indexes())
        current_left_free_seats = self._max_capacity - number_of_passengers

        if current_left_free_seats == 0:
            self._has_free_seats = False
        else:
            self._has_free_seats = True



    @property
    def speed(self) -> float:
        return self._speed

    @property
    def max_speed(self) -> float:
        return self._max_speed

    @property
    def max_capacity(self) -> int:
        return self._max_capacity

    @property
    def passengers_last_names(self) -> Iterable[str]:
        return iter(self.passengers_last_names)

    @property
    def has_free_seats(self) -> bool:
        return self._has_free_seats

    def get_passengers_off(self, names: Iterable[str]) -> None:
        for name in names:
            if name not in self._passengers_last_names_seats.values():
                self.error_function(f"{name} is not in the bus!")
                continue

            passenger_seat_index = tuple(self._passengers_last_names_seats.values()).index(name)
            self._passengers_last_names_seats[passenger_seat_index] = None

        self._check_values()

    def _get_free_seats_indexes(self) -> tuple[int]:
        return tuple(filter(lambda index: self._passengers_last_names_seats[index] is None, self._passengers_last_names_seats))

    def take_passengers(self, names: Iterable[str]) -> None:
        free_seats_indexes = list(self._get_free_seats_indexes())

        for name in names:
            if len(free_seats_indexes) <= 0:
                self.error_function(f"Seats are gone. {name} and other passengers after this name will not be taken.")
                break

            self._passengers_last_names_seats[free_seats_indexes[0]] = name
            free_seats_indexes.pop(0)

        self._check_values()

    def __add__(self, other: str) -> Self:
        self.take_passengers((other,))

        self._check_values()
        return self

    def __sub__(self, other: str) -> Self:
        self.get_passengers_off((other,))

        self._check_values()
        return self

    def __contains__(self, name: str) -> bool:
        return name in self._passengers_last_names_seats.values()

    def __str__(self) -> str:
        bus_data = f"""
Bus data:
    Current speed: {self.speed}/{self.max_speed}
    Has free seats: {self._has_free_seats}
    Passengers {self._max_capacity - len(self._get_free_seats_indexes())}/{self.max_capacity}.

Passengers:\n"""

        for index, name in self._passengers_last_names_seats.items():
            bus_data += f"{index}: {name}, "

        return bus_data



bus = Bus(passengers_last_names=["hello"] * 110)
print(bus)



Bus data:
    Current speed: 50/90
    Has free seats: True
    Passengers 100/100.

Passengers:
0: hello, 1: hello, 2: hello, 3: hello, 4: hello, 5: hello, 6: hello, 7: hello, 8: hello, 9: hello, 10: hello, 11: hello, 12: hello, 13: hello, 14: hello, 15: hello, 16: hello, 17: hello, 18: hello, 19: hello, 20: hello, 21: hello, 22: hello, 23: hello, 24: hello, 25: hello, 26: hello, 27: hello, 28: hello, 29: hello, 30: hello, 31: hello, 32: hello, 33: hello, 34: hello, 35: hello, 36: hello, 37: hello, 38: hello, 39: hello, 40: hello, 41: hello, 42: hello, 43: hello, 44: hello, 45: hello, 46: hello, 47: hello, 48: hello, 49: hello, 50: hello, 51: hello, 52: hello, 53: hello, 54: hello, 55: hello, 56: hello, 57: hello, 58: hello, 59: hello, 60: hello, 61: hello, 62: hello, 63: hello, 64: hello, 65: hello, 66: hello, 67: hello, 68: hello, 69: hello, 70: hello, 71: hello, 72: hello, 73: hello, 74: hello, 75: hello, 76: hello, 77: hello, 78: hello, 79: hello, 80: hello, 81: hello, 82: hello,

C:\Users\Liteop\AppData\Local\Temp\ipykernel_54052\3176360996.py:13: UserWarning: Number of passengers is bigger than capacity: 101/100. Extra passengers are gone.
  error_function(f"Number of passengers is bigger than capacity: {index}/{max_capacity}. Extra passengers are gone.")


In [68]:
# ChatGPT thanks for tests :)

def test_bus():
    # Создаем автобус с 5 местами и 3 пассажирами
    bus = Bus(passengers_last_names=["Smith", "Johnson", "Williams"], max_capacity=5)

    # Проверяем базовые параметры
    assert bus.speed == 50, "Начальная скорость должна быть 50"
    assert bus.max_speed == 90, "Максимальная скорость должна быть 90"
    assert bus.max_capacity == 5, "Максимальная вместимость должна быть 5"
    assert bus.has_free_seats is True, "Должны быть свободные места"
    assert "Smith" in bus, "Smith должен быть в автобусе"
    assert "Johnson" in bus, "Johnson должен быть в автобусе"
    assert "Williams" in bus, "Williams должен быть в автобусе"
    assert "Brown" not in bus, "Brown не должен быть в автобусе"

    # Добавляем пассажира
    bus += "Brown"
    assert "Brown" in bus, "Brown должен быть в автобусе после посадки"

    # Высаживаем пассажира
    bus -= "Smith"
    assert "Smith" not in bus, "Smith не должен быть в автобусе после высадки"

    # Проверяем, что не можем добавить больше пассажиров, чем мест
    bus += "Taylor"
    bus += "Anderson"
    bus += "Thomas"  # Этот не должен сесть, так как мест нет
    assert "Taylor" in bus, "Taylor должен быть в автобусе"
    assert "Anderson" in bus, "Anderson должен быть в автобусе"
    assert "Thomas" not in bus, "Thomas не должен быть в автобусе (мест нет)"

    print("Все тесты пройдены!")

# Запускаем тесты
test_bus()


Все тесты пройдены!


C:\Users\Liteop\AppData\Local\Temp\ipykernel_54052\3176360996.py:84: UserWarning: Seats are gone. Thomas and other passengers after this name will not be taken.
  self.error_function(f"Seats are gone. {name} and other passengers after this name will not be taken.")
